# Factor Exposures in Sports & Wellness Stocks

## Business Question

How do Nike, Garmin, Peloton and Planet Fitness differ in their exposure to broad market, consumer discretionary, and growth oriented factors?

## Why This Project 

I am extremely interested in the intersection of sports, fitness, business and analytics. These four public companies represent the different areas of the consumer wellness landscape, from wearables and apparel to connected gym operations. Rather than treating them as one category, I wanted to explore whether thier stock behavior reflects shared sector dynamics or meaningfully different business-model sensitivities.

## Project Scope 

This project is a short-horizon explanatory factor model, using daily returns from January 1, 2026 to April 7, 2026. This project was not intended for investment recommendation, but to demonstrate a reproducable workflow for comparing how sports and wellness companies relate to the broader market, consumer discretionary, and growth oriented factors. 

## Data

- Daily closing prices from **January 1, 2026 through April 7, 2026**
- Companies analyzed:
  - **Nike (NKE)**
  - **Garmin (GRMN)**
  - **Peloton (PTON)**
  - **Planet Fitness (PLNT)**
- Factors used:
  - **SPY** as a broad market proxy
  - **XLY** as a consumer discretionary proxy
  - **QQQ** as a growth-oriented proxy
- Source data was organized in Excel and then rebuilt in Python for a reproducible workflow

## Methodology

- Imported price data and converted dates into a consistent datetime format
- Calculated daily returns for each company and each factor
- Estimated simple one-factor linear regressions for each company against:
  - **SPY**
  - **XLY**
  - **QQQ**
- Compared factor betas across companies to evaluate relative sensitivity
- Calculated **R² vs SPY** to measure how much of each stock’s return variation was explained by broad market moves alone
- Rebased company price series to **100** at the start of the sample window to support normalized performance comparisons

## Results

The factor summary suggests that these companies do not behave as one clean category, even though they all sit within the broader sports and wellness space.

- **Nike** showed the strongest sensitivity to the broad market in this sample
- **Garmin** also displayed meaningful market sensitivity, with notable exposure to consumer and growth-oriented factors
- **Peloton** appeared especially sensitive to consumer discretionary and growth-oriented factors relative to its broad market beta
- **Planet Fitness** showed the lowest broad market sensitivity, suggesting more idiosyncratic behavior over this window

Across the group, consumer discretionary exposure was meaningful, but the differences in beta values suggest that business model still matters.

## Visual 1: Estimated Factor Betas by Company

![Estimated Factor Betas by Company](../docs/estimated_factor_betas.png)

## Visual 2: Normalized Stock Performance (Base = 100)

![Normalized Stock Performance](../docs/normalized_stock_performance.png)

## Key Takeaways

- Sports and wellness stocks do not necessarily move like a single unified segment
- **Consumer discretionary exposure** was relevant across all four names, which makes sense given their dependence on non-essential consumer spending
- **Broad market sensitivity** varied meaningfully, with Nike and Garmin appearing more market-linked than Peloton or Planet Fitness
- **Planet Fitness** stood out as the least explained by broad market movement alone
- Even within a related sector, factor exposure can differ substantially depending on whether a company is tied more closely to apparel, wearables, connected fitness, or gym operations

## Limitations

- This is a **short-horizon** analysis based on a limited daily-return window
- The model uses a **simple one-factor approach** for comparison rather than a full multi-factor regression
- Results are exploratory and should not be interpreted as investment advice
- Factor relationships may change over a longer sample period or under different market conditions


## Next Steps

- Extend the sample window to **6–12 months** for more stable estimates
- Test a **multi-factor regression** framework using all factors together
- Add **rolling betas** to see how factor exposure changes over time
- Compare these names against a custom **sports and wellness basket**
- Build an interactive Tableau dashboard to visualize factor exposure and normalized price performance



In [1]:
import sys
print(sys.executable)

/opt/miniconda3/envs/client_project/bin/python


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [5]:
df = pd.read_excel("../data/Factor_model_sports.xlsx")

/opt/miniconda3/envs/client_project/lib/python3.12/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [6]:
file_path = "../data/Factor_model_sports.xlsx"

prices = pd.read_excel(file_path, sheet_name="Prices")
returns = pd.read_excel(file_path, sheet_name="Returns")
summary_excel = pd.read_excel(file_path, sheet_name="Summary")

/opt/miniconda3/envs/client_project/lib/python3.12/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [7]:
prices["Date"] = pd.to_datetime(prices["Date"])
returns["Date"] = pd.to_datetime(returns["Date"])

In [8]:
companies = ["NKE Return", "GRMN Return", "PTON Return", "PLNT Return"]
factors = {
    "SPY": "SPY Return",
    "XLY": "XLY Return",
    "QQQ": "QQQ Return"
}

results = []

for company in companies:
    row = {"Company": company.replace(" Return", "")}
    
    for factor_name, factor_col in factors.items():
        x = returns[[factor_col]].dropna()
        y = returns.loc[x.index, company]
        
        model = LinearRegression()
        model.fit(x, y)
        y_pred = model.predict(x)
        
        row[f"Beta vs {factor_name}"] = model.coef_[0]
        
        if factor_name == "SPY":
            row["R^2 vs SPY"] = r2_score(y, y_pred)
    
    results.append(row)

summary_py = pd.DataFrame(results)
summary_py

,Company,Beta vs SPY,R^2 vs SPY,Beta vs XLY,Beta vs QQQ
0,NKE,0.779309,0.069233,0.703206,0.323237
1,GRMN,1.428732,0.328235,0.941041,1.031908
2,PTON,2.199897,0.162265,1.343068,1.585873
3,PLNT,0.225941,0.008826,0.046037,-0.017210


In [9]:
file_path = "../data/Factor_model_sports.xlsx"

prices = pd.read_excel(file_path, sheet_name="Prices")
returns = pd.read_excel(file_path, sheet_name="Returns")

prices["Date"] = pd.to_datetime(prices["Date"])
returns["Date"] = pd.to_datetime(returns["Date"])

/opt/miniconda3/envs/client_project/lib/python3.12/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [10]:
companies = ["NKE Return", "GRMN Return", "PTON Return", "PLNT Return"]
factors = {
    "SPY": "SPY Return",
    "XLY": "XLY Return",
    "QQQ": "QQQ Return"
}

results = []

for company in companies:
    row = {"Company": company.replace(" Return", "")}
    
    for factor_name, factor_col in factors.items():
        temp = returns[[company, factor_col]].dropna()
        x = temp[[factor_col]]
        y = temp[company]
        
        model = LinearRegression()
        model.fit(x, y)
        y_pred = model.predict(x)
        
        row[f"Beta vs {factor_name}"] = model.coef_[0]
        
        if factor_name == "SPY":
            row["R^2 vs SPY"] = r2_score(y, y_pred)
    
    results.append(row)

summary_py = pd.DataFrame(results)
summary_py

,Company,Beta vs SPY,R^2 vs SPY,Beta vs XLY,Beta vs QQQ
0,NKE,0.779309,0.069233,0.703206,0.323237
1,GRMN,1.428732,0.328235,0.941041,1.031908
2,PTON,2.199897,0.162265,1.343068,1.585873
3,PLNT,0.225941,0.008826,0.046037,-0.017210


In [11]:
company_price_cols = ["NKE: Close", "GRMN: Close", "PTON: Close", "PLNT: Close"]

normalized = prices[["Date"] + company_price_cols].copy()

for col in company_price_cols:
    normalized[col.replace(": Close", "")] = normalized[col] / normalized[col].iloc[0] * 100

normalized = normalized[["Date", "NKE", "GRMN", "PTON", "PLNT"]]
normalized.head()

,Date,NKE,GRMN,PTON,PLNT
0,2026-01-02,100.000000,100.000000,100.000000,100.000000
1,2026-01-05,101.975348,100.652045,101.633987,96.035724
2,2026-01-06,103.271176,103.413357,107.843137,96.172423
3,2026-01-07,99.905183,104.070342,107.843137,95.142623
4,2026-01-08,103.128951,105.443588,109.803922,96.382029


In [12]:
summary_display = summary_py.copy().round(3)

summary_py.to_csv("factor_summary_full.csv", index=False)
summary_display.to_csv("factor_summary_display.csv", index=False)
normalized.to_csv("normalized_prices.csv", index=False)

Format for Tableau

In [13]:
summary_long = summary_py.melt(
    id_vars="Company",
    value_vars=["Beta vs SPY", "Beta vs XLY", "Beta vs QQQ"],
    var_name="Factor",
    value_name="Beta"
)

summary_long.to_csv("factor_summary_long.csv", index=False)

In [14]:
normalized_long = normalized.melt(
    id_vars="Date",
    value_vars=["NKE", "GRMN", "PTON", "PLNT"],
    var_name="Company",
    value_name="Normalized Price"
)

normalized_long.to_csv("normalized_prices_long.csv", index=False)